# Human vs Machine: CalMS21 Task 1 Baseline

This notebook applies BANOS to a **human-vs-machine** annotation setting using the
CalMS21 Task 1 benchmark dataset.

**What we do:**
1. Load the CalMS21 Task 1 baseline model (Conv-1D trained on pose keypoints)
   and run inference on the 19 test recordings
2. Reproduce CalMS21's published pooled macro-F1, confirming we replicate
   their evaluation pipeline exactly
3. Show our preferred per-recording frame F1 + BANOS metrics side-by-side,
   explaining deliberate differences from CalMS21's approach

**Python-only** (no MATLAB equivalent) — requires `tensorflow`.

> For BANOS fundamentals, see [`tutorial_python.ipynb`](tutorial_python.ipynb).

**Install dependencies:**
```bash
# Option A — via the calms21 extra (recommended)
pip install "banos[calms21]"

# Option B — manually
pip install tensorflow scikit-learn tqdm
```

**Launch with uv (dev environment):**
```bash
uv run --extra calms21 --extra notebook jupyter notebook demo/tutorial_calms21_task1.ipynb
```

**Data:** The CalMS21 test JSON (~600 MB) is downloaded automatically on first run
and cached in `.cache/calms21/` (gitignored). The model weights are included in
the repo at `demo/models/task1_seed_42_model.h5`.

---
## Section 1 — Setup

In [1]:
import sys
import zipfile
from pathlib import Path
import math
import urllib.request

import numpy as np
import pandas as pd

try:
    import tensorflow as tf
    print(f"TensorFlow {tf.__version__}")
except ImportError:
    print("TensorFlow not found. Install with: pip install banos[calms21]")
    sys.exit(1)

import banos
from sklearn.metrics import f1_score

print(f"BANOS {banos.__version__}")


def _find_demo_dir():
    """Locate the demo/ directory regardless of where Jupyter was launched."""
    try:
        nb_file = Path(__file__).resolve()
        if nb_file.parent.name == "demo":
            return nb_file.parent
    except NameError:
        pass
    cwd = Path(".").resolve()
    for candidate in [cwd, cwd.parent]:
        if (candidate / "models" / "task1_seed_42_model.h5").exists():
            return candidate
    return cwd


DEMO_DIR = _find_demo_dir()

MODEL_PATH = DEMO_DIR / "models" / "task1_seed_42_model.h5"

# CalMS21 Task 1 classification zip (~457 MB) — contains the test JSON
TASK1_ZIP_URL  = "https://data.caltech.edu/records/s0vdx-0k302/files/task1_classic_classification.zip?download=1"
CACHE_DIR      = DEMO_DIR.parent / ".cache" / "calms21"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TASK1_ZIP_PATH = CACHE_DIR / "task1_classic_classification.zip"
TEST_JSON_PATH = CACHE_DIR / "calms21_task1_test.json"

BEHAVIORS = ["attack", "investigation", "mount"]
VOCAB     = {"attack": 0, "investigation": 1, "mount": 2, "other": 3}

assert MODEL_PATH.exists(), (
    f"Model not found at {MODEL_PATH}\n"
    f"Make sure you cloned the full repo (demo/models/task1_seed_42_model.h5 must exist)."
)
print(f"Model: {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1e3:.0f} KB)")
print(f"Cache: {CACHE_DIR}")

TensorFlow 2.21.0
BANOS 0.2.0
Model: C:\Users\GA401\Documents\Pro\Code\GithubPerso\Tl_BANOS_dev\demo\models\task1_seed_42_model.h5  (959 KB)
Cache: C:\Users\GA401\Documents\Pro\Code\GithubPerso\Tl_BANOS_dev\.cache\calms21


---
## Section 2 — Download CalMS21 Test Data

The CalMS21 Task 1 classification zip (~457 MB) is downloaded once from CaltechDATA,
then the test JSON is extracted from it. Both are cached in `.cache/calms21/` (gitignored).

If the download fails, download `task1_classic_classification.zip` manually from
https://data.caltech.edu/records/s0vdx-0k302 and place it at the path printed above.

In [2]:
def download_with_progress(url, dest_path, label):
    """Download a file once; skip if already present."""
    dest_path = Path(dest_path)
    if dest_path.exists():
        print(f"{label}: already cached ({dest_path.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"{label}: downloading from {url} ...")
    try:
        from tqdm import tqdm

        class _TqdmHook(tqdm):
            def update_to(self, b=1, bsize=1, tsize=None):
                if tsize is not None:
                    self.total = tsize
                self.update(b * bsize - self.n)

        with _TqdmHook(unit="B", unit_scale=True, miniters=1, desc=label) as t:
            urllib.request.urlretrieve(url, dest_path, reporthook=t.update_to)
    except ImportError:
        def _hook(count, block_size, total_size):
            if total_size > 0 and count % 500 == 0:
                pct = min(100.0, 100 * count * block_size / total_size)
                print(f"  {pct:.0f}%", end="\r")
        urllib.request.urlretrieve(url, dest_path, reporthook=_hook)
        print()
    print(f"{label}: done ({dest_path.stat().st_size / 1e6:.1f} MB)")


def extract_test_json(zip_path, dest_path):
    """Extract calms21_task1_test.json from the zip into dest_path."""
    if dest_path.exists():
        print(f"Test JSON: already extracted ({dest_path.stat().st_size / 1e6:.1f} MB)")
        return
    print("Extracting calms21_task1_test.json from zip ...")
    with zipfile.ZipFile(zip_path) as zf:
        # Find the test JSON entry (may be in a subdirectory inside the zip)
        matches = [n for n in zf.namelist() if n.endswith("calms21_task1_test.json")]
        if not matches:
            raise FileNotFoundError(
                "calms21_task1_test.json not found in zip. "
                f"Zip contains: {zf.namelist()[:10]} ..."
            )
        entry = matches[0]
        with zf.open(entry) as src, open(dest_path, "wb") as dst:
            import shutil
            shutil.copyfileobj(src, dst)
    print(f"Test JSON: extracted ({dest_path.stat().st_size / 1e6:.1f} MB)")


download_with_progress(TASK1_ZIP_URL, TASK1_ZIP_PATH, "CalMS21 task1 zip")
extract_test_json(TASK1_ZIP_PATH, TEST_JSON_PATH)

CalMS21 task1 zip: downloading from https://data.caltech.edu/records/s0vdx-0k302/files/task1_classic_classification.zip?download=1 ...


CalMS21 task1 zip: 457MB [00:41, 11.1MB/s]                                                                    


CalMS21 task1 zip: done (457.4 MB)
Extracting calms21_task1_test.json from zip ...
Test JSON: extracted (632.2 MB)


---
## Section 3 — Parse Test JSON

In [3]:
import json

print("Loading test JSON (this may take a moment for a 600 MB file)...")
with open(TEST_JSON_PATH) as f:
    raw = json.load(f)

# Structure: { 'annotator-id_0': { 'task1/test/<name>': { keypoints, annotations, ... } } }
seqs       = raw["annotator-id_0"]
recordings = sorted(seqs.keys())

print(f"Number of test recordings: {len(recordings)}")

vocab_check = seqs[recordings[0]]["metadata"]["vocab"]
assert vocab_check == VOCAB, f"Unexpected vocab: {vocab_check}"
print(f"Vocabulary confirmed: {VOCAB}")

dataset = {
    rec: {
        "keypoints": np.array(seqs[rec]["keypoints"]),   # (n_frames, 2, 2, 7)
        "gt":        np.array(seqs[rec]["annotations"]),  # (n_frames,) 4-class int
    }
    for rec in recordings
}

total_frames = sum(v["gt"].shape[0] for v in dataset.values())
print(f"Total frames: {total_frames:,}")
print()
for rec in recordings[:3]:
    n = dataset[rec]["gt"].shape[0]
    labels, counts = np.unique(dataset[rec]["gt"], return_counts=True)
    beh_str = "  ".join(f"{list(VOCAB)[l]}={int(c)}" for l, c in zip(labels, counts))
    print(f"  {rec.split('/')[-1]}: {n} frames | {beh_str}")
print("  ...")

Loading test JSON (this may take a moment for a 600 MB file)...
Number of test recordings: 19
Vocabulary confirmed: {'attack': 0, 'investigation': 1, 'mount': 2, 'other': 3}
Total frames: 262,107

  mouse071_task1_annotator1: 23810 frames | investigation=4679  mount=2894  other=16237
  mouse072_task1_annotator1: 18164 frames | investigation=4298  mount=6409  other=7457
  mouse073_task1_annotator1: 19156 frames | investigation=6867  mount=5050  other=7239
  ...


---
## Section 4 — Preprocess, Build Windows, and Run Inference

The CalMS21 baseline preprocessing pipeline:
1. **Transpose** `(n, 2, 2, 7)` → `(n, 2, 7, 2)` so the last axis is `(x, y)`
2. **Normalize** `x / 1024`, `y / 570` (original image resolution is 1024 × 570 px)
3. **Flatten** → `(n, 28)` — 2 mice × 7 body parts × 2 coords

Each frame then becomes a **201-frame window** (100 past × step 2, centre, 100 future × step 2).
Boundaries are zero-padded, matching the original training generator.

Windows are built one recording at a time to keep peak RAM usage ~500 MB.

In [4]:
def preprocess_keypoints(kp):
    """Normalize and flatten keypoints to shape (n_frames, 28)."""
    kp = kp.transpose(0, 1, 3, 2).astype(np.float32)  # (n, 2, 7, 2)
    kp[..., 0] /= 1024.0
    kp[..., 1] /= 570.0
    return kp.reshape(kp.shape[0], -1)                 # (n, 28)


def make_windows(kp_norm, past=100, future=100, gap=2):
    """
    Build sliding temporal windows matching the CalMS21 baseline data generator.

    kp_norm : ndarray (n_frames, 28)
    Returns : ndarray (n_frames, past + 1 + future, 28)  — i.e. (n, 201, 28)
    """
    n, feat_dim = kp_norm.shape
    left_pad  = past  * gap   # 200 zero frames on the left
    right_pad = future * gap  # 200 zero frames on the right
    padded = np.pad(kp_norm, ((left_pad, right_pad), (0, 0)))  # (n + 400, 28)

    # Vectorised index construction — no Python loop over frames
    offsets     = np.arange(0, (past + 1 + future) * gap, gap)  # (201,)
    all_indices = np.arange(n)[:, None] + offsets                # (n, 201)
    return padded[all_indices]  # (n, 201, 28)


# compile=False: avoids needing tensorflow-addons (used only for training metrics)
print("Loading model...")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Model loaded. Running inference on 19 test recordings...")
print()

for rec in recordings:
    kp_norm = preprocess_keypoints(dataset[rec]["keypoints"])   # (n, 28)
    X       = make_windows(kp_norm)                              # (n, 201, 28)
    probs   = model.predict(X, batch_size=512, verbose=0)        # (n, 4)
    dataset[rec]["pred"] = np.argmax(probs, axis=-1)             # (n,)
    print(f"  {rec.split('/')[-1]}: {len(probs):,} frames")
    del X, probs  # free window array before next recording

print("\nInference complete.")

Loading model...
Model loaded. Running inference on 19 test recordings...

  mouse071_task1_annotator1: 23,810 frames
  mouse072_task1_annotator1: 18,164 frames
  mouse073_task1_annotator1: 19,156 frames
  mouse074_task1_annotator1: 18,825 frames
  mouse075_task1_annotator1: 18,015 frames
  mouse076_task1_annotator1: 19,710 frames
  mouse077_task1_annotator1: 18,907 frames
  mouse078_task1_annotator1: 19,473 frames
  mouse079_task1_annotator1: 17,483 frames
  mouse080_task1_annotator1: 6,179 frames
  mouse081_task1_annotator1: 4,932 frames
  mouse082_task1_annotator1: 7,115 frames
  mouse083_task1_annotator1: 4,881 frames
  mouse084_task1_annotator1: 5,626 frames
  mouse085_task1_annotator1: 19,200 frames
  mouse086_task1_annotator1: 8,617 frames
  mouse087_task1_annotator1: 11,915 frames
  mouse088_task1_annotator1: 8,825 frames
  mouse089_task1_annotator1: 11,274 frames

Inference complete.


---
## Section 5 — Convert to Binary DataFrames

BANOS expects one binary column per behavior. We convert 4-class integer labels
to three binary columns (attack / investigation / mount).

In [5]:
banos_data = {}
for rec in recordings:
    gt_4   = dataset[rec]["gt"]
    pred_4 = dataset[rec]["pred"]
    gt_df   = pd.DataFrame({b: (gt_4   == VOCAB[b]).astype(int) for b in BEHAVIORS})
    pred_df = pd.DataFrame({b: (pred_4 == VOCAB[b]).astype(int) for b in BEHAVIORS})
    banos_data[rec] = (pred_df, gt_df)

rec0 = recordings[0]
p0, g0 = banos_data[rec0]
print(f"Example — {rec0.split('/')[-1]}")
print(f"  GT bout counts:   { {b: int((g0[b].diff().fillna(g0[b]) == 1).sum()) for b in BEHAVIORS} }")
print(f"  Pred bout counts: { {b: int((p0[b].diff().fillna(p0[b]) == 1).sum()) for b in BEHAVIORS} }")

Example — mouse071_task1_annotator1
  GT bout counts:   {'attack': 0, 'investigation': 79, 'mount': 26}
  Pred bout counts: {'attack': 1, 'investigation': 76, 'mount': 29}


---
## Section 6 — CalMS21 Official F1 (Pooled Method)

CalMS21 evaluates by **pooling all frames** from all recordings, then computing
per-class F1 with `sklearn.metrics.f1_score(average=None)`.
Macro-F1 = mean of attack / investigation / mount (indices 0, 1, 2);
"other" (index 3) is excluded.

We replicate this exactly to confirm our model and pipeline are correct.

In [6]:
all_gt   = np.concatenate([dataset[r]["gt"]   for r in recordings])
all_pred = np.concatenate([dataset[r]["pred"] for r in recordings])

f1_per_class = f1_score(all_gt, all_pred, average=None,
                        labels=[0, 1, 2, 3], zero_division=0)

calms21_macro_f1 = f1_per_class[:3].mean()   # exclude "other"

print("CalMS21 Official F1 (pooled frames, sklearn, macro excluding 'other'):")
for beh, f1v in zip(BEHAVIORS, f1_per_class[:3]):
    print(f"  {beh:<16} F1 = {f1v:.4f}")
print(f"  {'other':<16} F1 = {f1_per_class[3]:.4f}  (excluded)")
print(f"\n  Macro F1 (excl. other): {calms21_macro_f1:.4f}")

CalMS21 Official F1 (pooled frames, sklearn, macro excluding 'other'):
  attack           F1 = 0.6318
  investigation    F1 = 0.8057
  mount            F1 = 0.9000
  other            F1 = 0.9456  (excluded)

  Macro F1 (excl. other): 0.7792


---
## Section 7 — Our Per-Recording Frame F1

We compute frame F1 **per recording, per behavior**, macro-average across
behaviors, then mean across recordings.

**Two deliberate differences from CalMS21:**
1. **Per-recording aggregation** — treats each recording equally (CalMS21 weights by length)
2. **Explicit absent-behavior scoring**  
   - GT = 0 bouts AND machine = 0 bouts → **1.0** (correct absence)  
   - GT = 0 bouts BUT machine predicts bouts → **0.0** (false detection)

In [7]:
def frame_f1(pred_col, gt_col):
    """Frame-level F1 with explicit absent-behavior scoring."""
    gt_col, pred_col = np.asarray(gt_col), np.asarray(pred_col)
    if gt_col.sum() == 0 and pred_col.sum() == 0:
        return 1.0
    if gt_col.sum() == 0 and pred_col.sum() > 0:
        return 0.0
    tp = int(((pred_col == 1) & (gt_col == 1)).sum())
    fp = int(((pred_col == 1) & (gt_col == 0)).sum())
    fn = int(((pred_col == 0) & (gt_col == 1)).sum())
    p  = tp / (tp + fp) if tp + fp > 0 else float("nan")
    r  = tp / (tp + fn) if tp + fn > 0 else float("nan")
    return 2 * p * r / (p + r) if (p + r) > 0 else float("nan")


per_rec_f1 = {}
for rec in recordings:
    p_df, g_df = banos_data[rec]
    beh_f1s = {b: frame_f1(p_df[b].values, g_df[b].values) for b in BEHAVIORS}
    per_rec_f1[rec] = {**beh_f1s, "macro": np.nanmean(list(beh_f1s.values()))}

our_frame_f1 = np.nanmean([v["macro"] for v in per_rec_f1.values()])

print("Per-recording Frame F1 (mean across recordings):")
for b in BEHAVIORS:
    beh_mean = np.nanmean([per_rec_f1[r][b] for r in recordings])
    print(f"  {b:<16} mean F1 = {beh_mean:.4f}")
print(f"\n  Overall (macro of means): {our_frame_f1:.4f}")

Per-recording Frame F1 (mean across recordings):
  attack           mean F1 = 0.6325
  investigation    mean F1 = 0.7441
  mount            mean F1 = 0.8085

  Overall (macro of means): 0.7267


---
## Section 8 — BANOS Metrics

In [8]:
group_metrics, overall_metrics = banos.score(banos_data)

print("BANOS per-behavior metrics (nanmean across 19 recordings):")
group_df = pd.DataFrame(group_metrics).T[
    ["f1_score", "so", "tp", "ic", "precision", "recall"]
]
group_df.columns = ["DA (F1)", "SO", "TP", "IC", "Precision", "Recall"]
print(group_df.round(4).to_string())

print("\nBANOS overall metrics (nanmean across behaviors and recordings):")
for metric, value in overall_metrics.items():
    v = f"{value:.4f}" if (value is not None and not math.isnan(value)) else "NaN"
    print(f"  {metric:<12} {v}")

BANOS per-behavior metrics (nanmean across 19 recordings):
               DA (F1)      SO      TP      IC  Precision  Recall
attack          0.5745  0.5272  0.4800  0.7287     0.5548  0.6076
investigation   0.6476  0.4556  0.0651  0.9734     0.6166  0.7061
mount           0.7006  0.6357  0.2586  0.9374     0.6801  0.7264

BANOS overall metrics (nanmean across behaviors and recordings):
  precision    0.6172
  recall       0.6801
  f1_score     0.6398
  so           0.5378
  tp           0.2681
  ic           0.8798


---
## Section 9 — Side-by-Side Comparison

### Why the numbers differ

| Method | Aggregation | Absent behavior |
|--------|-------------|------------------|
| **CalMS21 macro-F1** | Pool all frames globally | Implicit (rare behaviors diluted into pool) |
| **Our frame F1** | Per recording → mean | Explicit: correct absence = 1.0, false detection = 0.0 |
| **BANOS DA** | Per recording → nanmean | Same as our frame F1 |

BANOS additionally reveals **how** the model errs:
- Low **SO** → predicted bouts have poor temporal overlap with GT bouts
- Low **TP** → bout boundaries (start/end frames) are imprecise
- High **IC** → predictions are stable once they fire (no within-bout flickering)

In [9]:
banos_da = overall_metrics["f1_score"]
banos_so = overall_metrics["so"]
banos_tp = overall_metrics["tp"]
banos_ic = overall_metrics["ic"]


def _fmt(v):
    return f"{v:.4f}" if (v is not None and not math.isnan(v)) else "NaN"


comparison = [
    ("CalMS21 F1 (pooled)",  calms21_macro_f1,
     "Global frame overlap; matches leaderboard method"),
    ("Our Frame F1",         our_frame_f1,
     "Per-recording avg; explicit absent-behavior scoring"),
    ("BANOS DA (F1)",        banos_da,
     "Bout-level detection accuracy"),
    ("BANOS SO",             banos_so,
     "Temporal overlap quality of matched bouts"),
    ("BANOS TP",             banos_tp,
     "Boundary precision (start/end frame accuracy)"),
    ("BANOS IC",             banos_ic,
     "Label stability within detected bouts"),
]

print(f"{'Metric':<24} {'Value':>7}  What it captures")
print("-" * 75)
for name, val, desc in comparison:
    print(f"{name:<24} {_fmt(val):>7}  {desc}")

Metric                     Value  What it captures
---------------------------------------------------------------------------
CalMS21 F1 (pooled)       0.7792  Global frame overlap; matches leaderboard method
Our Frame F1              0.7267  Per-recording avg; explicit absent-behavior scoring
BANOS DA (F1)             0.6398  Bout-level detection accuracy
BANOS SO                  0.5378  Temporal overlap quality of matched bouts
BANOS TP                  0.2681  Boundary precision (start/end frame accuracy)
BANOS IC                  0.8798  Label stability within detected bouts
